## Atividade prática 2
Segurança, extração de PDFs e exploração de dados de vacinação

*Aluna: Beatriz Goulart*

In [ ]:
!pip install pdfplumber pandas -q

### Autenticação segura com Colab Secrets

Para evitar a exposição de chaves de API e tokens no código fonte, utilizamos o gerenciador nativo de **Secrets** do Google Colab.

O token foi armazenado sob a variável `TOKEN_API`.

In [ ]:
from google.colab import userdata

api_token = userdata.get('TOKEN_API')

Extração de Dados com `pdfplumber`




In [ ]:
import pdfplumber
import pandas as pd

pdf_path = "remume_versao_2024.pdf"
dados_remume = []

with pdfplumber.open(pdf_path) as pdf:
    for pagina in pdf.pages:
        tabela = pagina.extract_table()
        if tabela:
            if not dados_remume:
                dados_remume.extend(tabela)
            else:
                dados_remume.extend(tabela[1:])

num_colunas_esperado = len(dados_remume[0])
linhas_normalizadas = []

for linha in dados_remume[1:]:
    if len(linha) == num_colunas_esperado:
        linhas_normalizadas.append(linha)
    elif len(linha) > num_colunas_esperado:
        linhas_normalizadas.append(linha[:num_colunas_esperado])
    else:
        linha_expandida = linha + [None] * (num_colunas_esperado - len(linha))
        linhas_normalizadas.append(linha_expandida)

df_remume = pd.DataFrame(linhas_normalizadas, columns=dados_remume[0])


### Tratamento e higienização dos dados

Aplicação de filtros para eliminar registros vazios e normalizar o espaçamento dos campos de texto, preparando a base para análise.

In [ ]:
# Remoção de eventuais linhas completamente nulas
df_remume.dropna(how='all', inplace=True)

df_remume = df_remume.applymap(lambda x: x.replace('\n', ' ') if isinstance(x, str) else x)

df_remume.head()

/tmp/ipykernel_11615/2423050865.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_remume = df_remume.applymap(lambda x: x.replace('\n', ' ') if isinstance(x, str) else x)


,Item,Medicamento,Apresentação,Local de acesso,None,Classificação e tipo de\nreceita
0,01,Acetilcisteína,600 mg granulado envelopes com 5 g,- Farmácias Distritais,None,- Expectorante - Receituário simples
1,02,Aciclovir,200 mg Comprimido,- Farmácias Distritais - Farmácia Estratégica,None,- Antiviral - Receituário simples
2,03,Aciclovir,50 mg/g Creme bisnaga,,- Farmácias Distritais,- Antiviral - Receituário simples
3,None,None,None,None,- Farmácia Estratégica,None
4,None,None,None,None,- Dispensários nas UBS,None


####  Análise dos dados extraídos

Filtro no dataframe para extrair e contabilizar os medicamentos cujo acesso se dá por meio de **Receituário Simples** nas unidades de saúde.

In [ ]:
coluna_localizada = None

for col in df_remume.columns:
    if 'classificação' in str(col).lower():
        coluna_localizada = col
        break

if coluna_localizada:
    df_simples = df_remume[df_remume[coluna_localizada].str.contains('simples', case=False, na=False)]
    print(f"[SUCESSO] Coluna identificada como: '{coluna_localizada}'")
    print(f"Total de medicamentos sob regulamentação de Receita Simples: {len(df_simples)}\n")

    colunas_exibicao = ['Medicamento', 'Apresentação']
    if 'Local de acesso' in df_remume.columns:
        colunas_exibicao.append('Local de acesso')

    print(df_simples[colunas_exibicao].head())
else:
    print(" Não foi possível encontrar nenhuma coluna de Classificação. Nomes das colunas disponíveis:")
    print(list(df_remume.columns))

[SUCESSO] Coluna identificada como: 'Classificação e tipo de
receita'
Total de medicamentos sob regulamentação de Receita Simples: 17

              Medicamento                        Apresentação  \
0          Acetilcisteína  600 mg granulado envelopes com 5 g   
1               Aciclovir                   200 mg Comprimido   
2               Aciclovir               50 mg/g Creme bisnaga   
5  Ácido acetilsalicílico                   100 mg Comprimido   
6            Ácido fólico                     5 mg Comprimido   

                                     Local de acesso  
0                             - Farmácias Distritais  
1      - Farmácias Distritais - Farmácia Estratégica  
2                                                     
5      - Farmácias Distritais - Dispensários nas UBS  
6  - Farmácias Distritais - Farmácia Estratégica ...  


#### Exportação da base de dados tratada

Com os dados do PDF completamente extraídos, normalizados e limpos, realizamos a exportação do dataframe para um arquivo no formato `.csv`. Esse arquivo estruturado poderá ser compartilhado ou utilizado em ferramentas para futuras tomadas de decisão na gestão de saúde municipal.

In [ ]:
nome_arquivo = "remume_criciuma_tratado.csv"
df_remume.to_csv(nome_arquivo, index=False, encoding="utf-8-sig")

print(f"Arquivo '{nome_arquivo}' gerado.")

Arquivo 'remume_criciuma_tratado.csv' gerado.
